# Appendix C.3 — WINEP (PR24 Water Industry National Environment Programme)

Evidence for the three ambiguities cited in Appendix C, plus the smaller ones noted there.

**Sources used**

| file | what it is |
| --- | --- |
| `raw_datasets/PR24 WINEP National Dataset.xlsx` | the national WINEP submission, 18,598 rows |
| `raw_datasets/access_database_csv_files/determinands.csv` | the permit register, for what a limit attaches to |
| `ttl/winep.ttl` | the delivered graph |

The demonstrator keeps Wessex Water × Water Quality × actions proposing a limit × the Poole Harbour
catchment — **11 actions**. Every ambiguity below is visible in those 11 rows unless stated otherwise.


In [1]:
import os, json, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "raw_datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
RAW = ROOT / "raw_datasets"
REG = RAW / "access_database_csv_files"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
print("repository root:", ROOT)


repository root: /Users/waf/git/projects/demonstrator-poc


In [2]:
import openpyxl

wb = openpyxl.load_workbook(RAW / "PR24 WINEP National Dataset.xlsx", read_only=True, data_only=True)
ws = wb["PR24 WINEP National Data"]
rows = ws.iter_rows(values_only=True)
header = list(next(rows))
winep = pd.DataFrame(list(rows), columns=header)
print(f"WINEP national dataset: {len(winep):,} rows x {len(header)} columns")

wq = winep[(winep.Water_Company == "Wessex Water Service Ltd") & (winep.EA_Function == "Water Quality")].copy()
wq["permit"] = wq.Licence_Permit_Obstruction_ID.astype(str).str.strip().apply(
    lambda r: r.zfill(6) if r.isdigit() else r)
print(f"Wessex Water x Water Quality:  {len(wq):,} rows")

PROPOSED = [c for c in winep.columns if c.startswith("Proposed_") and "DWF" not in c]
print(f"\nThe {len(PROPOSED)} proposed-limit columns:")
for c in PROPOSED:
    print("   ", c)


WINEP national dataset: 18,598 rows x 83 columns
Wessex Water x Water Quality:  1,233 rows

The 10 proposed-limit columns:
    Proposed_BOD_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable
    Proposed_NH3_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable
    Proposed_P_annual_average_permit_mg/l_(S=summer)/backstop_limit(mg/l)
    Proposed_P_95%ile_permit_(mg/l)
    Proposed_P_stretch_permit_targets_annual_average(mg/l)
    Proposed_Iron_/_Aluminium_permit_limits_(ug/l)
    Proposed_Chemical_permit_99%ile_LUT_(ug/l)
    Proposed_Chemical_permit_95%ile_LUT_(ug/l)
    Proposed_Chemical_annual_average_permit_conditions_(ug/l)
    Proposed_permit_other


In [3]:
import pyoxigraph as ox

store = ox.Store()
store.bulk_load(path=str(ROOT / "ttl" / "regulation.ttl"), format=ox.RdfFormat.TURTLE)
catchment_permits = {
    str(r[0].value).rsplit("/", 1)[-1].replace("%2F", "/")
    for r in store.query("SELECT ?p WHERE { ?p a <http://environment.data.gov.uk/ontology/water/WaterDischargePermit> }")
}
scoped = wq[wq.permit.isin(catchment_permits)]
has_limit = scoped[scoped[PROPOSED].apply(
    lambda r: r.notna() & r.astype(str).str.strip().ne(""), axis=1).any(axis=1)]

short = {c: c.replace("Proposed_", "").replace("_permit", "")[:20] for c in PROPOSED}
print(f"actions in the delivered graph: {has_limit.Action_ID.nunique()}\n")
has_limit[["Action_ID","Action_Name","Driver_Code_Primary","permit","Completion_Date"] + PROPOSED].rename(
    columns=short).reset_index(drop=True)


actions in the delivered graph: 11



,Action_ID,Action_Name,Driver_Code_Primary,permit,Completion_Date,BOD_95%ile(mg/l)(S=S,NH3_95%ile(mg/l)(S=S,P_annual_average_mg/,P_95%ile_(mg/l),P_stretch_targets_an,Iron_/_Aluminium_lim,Chemical_99%ile_LUT_,Chemical_95%ile_LUT_,Chemical_annual_aver,permit_other
0,08WW102103,Blackheath WRC - Phosphorus & Nitrogen Removal,HD_IMP_NN,042451,2030-03-31,,,0.25,,,Fe 4mg/l 95%ile 8mg/l Max.,,,,N 10mg/l
1,08WW102104,Dorchester WRC - Phosphorus & Nitrogen Removal,HD_IMP_NN,401050,2030-03-31,,,0.25,,,No change from current,,,,N 10mg/l
2,08WW102105,Lytchett Minster WRC - Phosphorus & Nitrogen Removal,HD_IMP_NN,401242,2030-03-31,,,0.25,,,TBC,,,,N 10mg/l
3,08WW102106,Milborne St Andrew WRC - Phosphorus Removal,SSSI_IMP,042116,2030-03-31,,,0.3,,,Fe 4mg/l 95%ile 8mg/l Max.,,,,
4,08WW102107,Poole WRC - Phosphorus & Nitrogen Removal,HD_IMP_NN,401354,2030-03-31,,,0.25,,,No change from current,,,,N 5mg/l
5,08WW102108,Wareham WRC - Phosphorus & Nitrogen Removal,HD_IMP_NN,401336,2030-03-31,,,0.25,,,TBC,,,,N 10mg/l
6,08WW102109,Wool WRC - Phosphorus & Nitrogen Removal,HD_IMP_NN,401747,2030-03-31,,,0.25,,,No change from current,,,,N 10mg/l
7,08WW102200,Dorchester WRC - Nitrogen Permit (UWWTR),U_IMP1,401050,2030-03-31,,,,,,,,,,N 15mg/l
8,08WW102201,Dorchester WRC - Phosphorus Permit (UWWTR),U_IMP2,401050,2030-05-13,,,2,,,No change from current,,,,
9,08WW102202,Wool WRC - Phosphorus Permit (UWWTR),U_IMP1,401747,2030-05-13,,,2,,,No change from current,,,,


That table is the whole of WINEP as delivered. Read across it: every ambiguity in Appendix C is in
there. The sections below isolate each one.


---
## C.3.1 A limit is proposed against a permit; conditions are held per outlet and per version

> *"WINEP identifies the target as a permit, while the register keys conditions at (permit, version,
> outlet, effluent)."*

Blackheath WRC, permit `042451`. WINEP proposes an ammonia limit of `4.2 UT 16` against the permit:


In [4]:
nh3_col = [c for c in PROPOSED if c.startswith("Proposed_NH3")][0]
blackheath = has_limit[has_limit.permit == "042451"]
blackheath[["Action_ID","Action_Name","Driver_Code_Primary","permit","Completion_Date", nh3_col]]


,Action_ID,Action_Name,Driver_Code_Primary,permit,Completion_Date,Proposed_NH3_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable
16483,08WW102103,Blackheath WRC - Phosphorus & Nitrogen Removal,HD_IMP_NN,042451,2030-03-31,
16942,08WW100250,Blackheath WRC - Permit Change,WFD_ND,042451,2030-03-31,4.2 UT 16


In [5]:
det = pd.read_csv(REG / "determinands.csv", dtype=str, low_memory=False)
sw = det[det.EA_REGION == "SW"]
current = sw[(sw.PERMIT_REF == "042451") & (sw.DETE_CODE == "0111") & (sw.VERSION == "8")]
print("What the register holds for ammonia on that permit's current version:")
current[["PERMIT_REF","VERSION","OUTLET_NUMBER","EFFLUENT_NUMBER","DETE_CODE","DETE",
         "CODE_1","VAL_1","CODE_2","VAL_2","UNITS"]].sort_values("OUTLET_NUMBER")


What the register holds for ammonia on that permit's current version:


,PERMIT_REF,VERSION,OUTLET_NUMBER,EFFLUENT_NUMBER,DETE_CODE,DETE,CODE_1,VAL_1,CODE_2,VAL_2,UNITS
401956,042451,8,1a,1,0111,Ammoniacal Nitrogen as N,MAXIMUM VALUE,27,95 PERCENTILE,7,MILLIGRAM PER LITRE
431868,042451,8,1b,1,0111,Ammoniacal Nitrogen as N,MAXIMUM VALUE,27,95 PERCENTILE,7,MILLIGRAM PER LITRE
401960,042451,8,2a,1,0111,Ammoniacal Nitrogen as N,MAXIMUM VALUE,27,95 PERCENTILE,7,MILLIGRAM PER LITRE
381907,042451,8,2b,1,0111,Ammoniacal Nitrogen as N,MAXIMUM VALUE,27,95 PERCENTILE,7,MILLIGRAM PER LITRE


**Four outlets, four ammonia conditions.** WINEP names the permit and nothing else, so which of the
four the proposed 4.2 mg/l replaces is not recoverable from the dataset. Nor does WINEP name a permit
**version** — and as Appendix C.1.5 shows, 26 of this catchment's permit versions have no published
effective date, so "the current one" is not resolvable either.

The same gap governs the carried-over limits. `No change from current` continues a condition, without
saying which:


In [6]:
iron_col = [c for c in PROPOSED if c.startswith("Proposed_Iron")][0]
carried = has_limit[has_limit[iron_col].astype(str).str.contains("No change", na=False)]
carried[["Action_ID","Action_Name","permit", iron_col]]


,Action_ID,Action_Name,permit,Proposed_Iron_/_Aluminium_permit_limits_(ug/l)
16484,08WW102104,Dorchester WRC - Phosphorus & Nitrogen Removal,401050,No change from current
16487,08WW102107,Poole WRC - Phosphorus & Nitrogen Removal,401354,No change from current
16489,08WW102109,Wool WRC - Phosphorus & Nitrogen Removal,401747,No change from current
16492,08WW102201,Dorchester WRC - Phosphorus Permit (UWWTR),401050,No change from current
16493,08WW102202,Wool WRC - Phosphorus Permit (UWWTR),401747,No change from current


In [7]:
print("Permit 401050, its current version, across all outlets:")
p401050 = sw[(sw.PERMIT_REF == "401050") & (sw.VERSION == "5")]
p401050[["OUTLET_NUMBER","EFFLUENT_NUMBER","DETE_CODE","DETE","CODE_1","VAL_1","UNITS"]].sort_values(
    ["OUTLET_NUMBER","DETE_CODE"])


Permit 401050, its current version, across all outlets:


,OUTLET_NUMBER,EFFLUENT_NUMBER,DETE_CODE,DETE,CODE_1,VAL_1,UNITS
430098,1,1,0061,pH,MINIMUM VALUE,6,PH UNITS
370288,1,1,0085,BOD : 5 Day ATU,95 PERCENTILE,15,MILLIGRAM PER LITRE
440228,1,1,0111,Ammoniacal Nitrogen as N,95 PERCENTILE,5,MILLIGRAM PER LITRE
390266,1,1,0135,"Solids, Suspended at 105 C",95 PERCENTILE,30,MILLIGRAM PER LITRE
440229,1,1,0348,"Phosphorus, Total as P",MEAN VALUE,1,MILLIGRAM PER LITRE
407979,1,1,3647,Flow : To full treatment (Before spill to storm tank/overflow),MINIMUM VALUE,195,LITRE PER SECOND
430099,1,1,6051,Iron,MAXIMUM VALUE,7000,MICROGRAM PER LITRE
400940,1,1,7782,Flow : Dry Weather :- {DWF},MAXIMUM VALUE,9450,CUBIC METRE PER DAY
420012,2,1,8174,Weir Setting,MINIMUM VALUE,244,LITRE PER SECOND
400430,3,1,8174,Weir Setting,MINIMUM VALUE,381,LITRE PER SECOND


Here the substance happens to disambiguate — only outlet 1 carries an iron condition — so the
demonstrator can resolve it. Nothing in the *format* guarantees that, and Blackheath above shows the
case where it does not hold. The delivered graph therefore records what it continued, explicitly:


In [8]:
wstore = ox.Store()
wstore.bulk_load(path=str(ROOT / "ttl" / "winep.ttl"), format=ox.RdfFormat.TURTLE)
cont = pd.DataFrame(
    [[str(v.value).replace("http://example.com/water-regulation/", "") for v in r]
     for r in wstore.query("""PREFIX reg: <http://environment.data.gov.uk/ontology/regulation/>
        SELECT ?limit ?condition WHERE { ?limit reg:continuesCondition ?condition } ORDER BY ?limit""")],
    columns=["carried-over limit", "condition it continues"])
cont


,carried-over limit,condition it continues
0,action/08WW102104/limit/6051,permit/401050/version/5/outlet/1/effluent/1/condition/6051
1,action/08WW102107/limit/6051,permit/401354/version/9/outlet/1/effluent/1/condition/6051
2,action/08WW102107/limit/6057,permit/401354/version/9/outlet/1/effluent/1/condition/6057
3,action/08WW102109/limit/6051,permit/401747/version/3/outlet/1/effluent/1/condition/6051
4,action/08WW102201/limit/6051,permit/401050/version/5/outlet/1/effluent/1/condition/6051
5,action/08WW102202/limit/6051,permit/401747/version/3/outlet/1/effluent/1/condition/6051


---
## C.3.2 The proposed-limit cells are human-authored free text with an undefined vocabulary

> *"Meaning is split between the column header, which fixes substance, unit and statistic, and the cell
> contents, which may override any of them."*

Every distinct cell value Wessex wrote into a proposed-limit column, with the column that was supposed
to define it:


In [9]:
cells = pd.concat([
    wq[[c]].dropna().rename(columns={c: "cell"}).assign(column=c) for c in PROPOSED
])
cells["cell"] = cells.cell.astype(str).str.strip()
cells = cells[cells.cell != ""]
print(f"{len(cells)} non-empty proposed-limit cells across Wessex Water, "
      f"{cells.cell.nunique()} distinct values\n")
cells.groupby("cell").agg(times=("cell", "size"),
                          columns=("column", lambda s: " | ".join(sorted(set(s))[:1]))
                          ).sort_values("times", ascending=False).head(20)


266 non-empty proposed-limit cells across Wessex Water, 65 distinct values



,times,columns
cell,,
No change from current,66,Proposed_Iron_/_Aluminium_permit_limits_(ug/l)
0.25,63,Proposed_P_annual_average_permit_mg/l_(S=summer)/backstop_limit(mg/l)
Fe 4mg/l 95%ile 8mg/l Max.,34,Proposed_Iron_/_Aluminium_permit_limits_(ug/l)
1,10,Proposed_P_annual_average_permit_mg/l_(S=summer)/backstop_limit(mg/l)
0.5,5,Proposed_P_annual_average_permit_mg/l_(S=summer)/backstop_limit(mg/l)
0.4,5,Proposed_P_annual_average_permit_mg/l_(S=summer)/backstop_limit(mg/l)
N 10mg/l,5,Proposed_permit_other
0.3,5,Proposed_P_annual_average_permit_mg/l_(S=summer)/backstop_limit(mg/l)
TBC,4,Proposed_Iron_/_Aluminium_permit_limits_(ug/l)


In [10]:
patterns = {
    "tiered, 'UT' undefined":            r"\bUT\b",
    "'(upper tier)' annotation":         r"upper tier",
    "'Max.' -- absolute? 100%ile?":      r"Max",
    "a MASS LOAD, not a concentration":  r"kg/d",
    "undecided at source":               r"^TBC$",
    "inline analyte + inline unit":      r"^[A-Z][a-z]? ?\d",
}
for label, pat in patterns.items():
    hit = cells[cells.cell.str.contains(pat, case=False, regex=True, na=False)].drop_duplicates("cell")
    print(f"--- {label}  ({len(hit)} distinct) ---")
    print(hit.head(4).to_string(index=False), "\n")


--- tiered, 'UT' undefined  (10 distinct) ---
     cell                                                                                column
8.5 UT 50 Proposed_BOD_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable
  8 UT 30 Proposed_NH3_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable
4.2 UT 16 Proposed_NH3_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable
  2 UT 10 Proposed_NH3_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable 

--- '(upper tier)' annotation  (12 distinct) ---
                        cell                column
 0.0019306 (upper tier ug/l) Proposed_permit_other
0.00069640 (upper tier ug/l) Proposed_permit_other
 0.0064618 (upper tier ug/l) Proposed_permit_other
    0.0009 (upper tier ug/l) Proposed_permit_other 

--- 'Max.' -- absolute? 100%ile?  (2 distinct) ---
                                   cell                                         column
             Fe 4mg/l 95%ile

### The unit contradiction, in the delivered catchment

The iron/aluminium column declares **µg/l** in its own header. The cell writes **mg/l** — a factor of
1,000. Nothing arbitrates; the parser must decide which to believe.


In [11]:
print("column header:", iron_col)
print()
fe = has_limit[has_limit[iron_col].astype(str).str.contains("mg/l", na=False)]
print(fe[["Action_ID","Action_Name","permit", iron_col]].to_string(index=False))
print()
print("The cell says 4 mg/l. The column header says ug/l. They differ by a factor of 1,000.\n")
print("How the register writes an iron limit, for the catchment permits that carry one:")
print(sw[(sw.DETE_CODE == "6051") & (sw.PERMIT_REF.isin(["401050","401354","401747"]))][
    ["PERMIT_REF","VERSION","DETE_CODE","DETE","CODE_1","VAL_1","UNITS"]].drop_duplicates(
    ["PERMIT_REF","VAL_1"]).to_string(index=False))
print("\n-> microgrammes, as the WINEP column header also says. Read in the column's unit the")
print("   proposal is 4 ug/l; read in the cell's unit it is 4,000 ug/l. Only one can be right.")


column header: Proposed_Iron_/_Aluminium_permit_limits_(ug/l)

 Action_ID                                    Action_Name permit Proposed_Iron_/_Aluminium_permit_limits_(ug/l)
08WW102103 Blackheath WRC - Phosphorus & Nitrogen Removal 042451                     Fe 4mg/l 95%ile 8mg/l Max.
08WW102106    Milborne St Andrew WRC - Phosphorus Removal 042116                     Fe 4mg/l 95%ile 8mg/l Max.

The cell says 4 mg/l. The column header says ug/l. They differ by a factor of 1,000.

How the register writes an iron limit, for the catchment permits that carry one:
PERMIT_REF VERSION DETE_CODE DETE        CODE_1 VAL_1               UNITS
    401050       1      6051 Iron MAXIMUM VALUE  7000 MICROGRAM PER LITRE
    401747       3      6051 Iron MAXIMUM VALUE  4000 MICROGRAM PER LITRE
    401354       8      6051 Iron MAXIMUM VALUE  3000 MICROGRAM PER LITRE

-> microgrammes, as the WINEP column header also says. Read in the column's unit the
   proposal is 4 ug/l; read in the cell's unit it i

### A statistic that is never stated

The nitrogen proposals live in the free-text `Proposed_permit_other` column, which fixes no statistic at
all. The register's nitrogen limit for the same permit *is* statistic-bearing:


In [12]:
other = [c for c in PROPOSED if c.endswith("permit_other")][0]
print("WINEP proposes:")
print(has_limit[has_limit[other].notna() & has_limit[other].astype(str).str.strip().ne("")][
    ["Action_ID","permit", other]].to_string(index=False))
print("\nThe register says, for permit 401354:")
print(sw[(sw.PERMIT_REF == "401354") & (sw.DETE_CODE == "9686")][
    ["PERMIT_REF","VERSION","CODE_1","VAL_1","UNITS"]].tail(1).to_string(index=False))
print("\n'N 5mg/l' names no statistic. Is it a mean, a 95th percentile, an absolute maximum?")
print("The register's 10 mg/l is a MEAN VALUE. The two cannot be compared without assuming one.")


WINEP proposes:
 Action_ID permit Proposed_permit_other
08WW102103 042451              N 10mg/l
08WW102104 401050              N 10mg/l
08WW102105 401242              N 10mg/l
08WW102107 401354               N 5mg/l
08WW102108 401336              N 10mg/l
08WW102109 401747              N 10mg/l
08WW102200 401050              N 15mg/l

The register says, for permit 401354:
PERMIT_REF VERSION     CODE_1 VAL_1               UNITS
    401354       8 MEAN VALUE    10 MILLIGRAM PER LITRE

'N 5mg/l' names no statistic. Is it a mean, a 95th percentile, an absolute maximum?
The register's 10 mg/l is a MEAN VALUE. The two cannot be compared without assuming one.


### Two more, for completeness

The mass-load cell is real but falls outside this catchment — it is Wessex-wide, and worth flagging to
the dataset owner regardless:


In [13]:
load = wq[wq[other].astype(str).str.contains("kg/d", na=False)]
print("The mass-load cell, and where it sits:")
print(load[["Action_ID","Action_Name","permit","Driver_Code_Primary", other]].to_string(index=False))
print("\nin the Poole Harbour catchment?", bool(set(load.permit) & catchment_permits))


The mass-load cell, and where it sits:
 Action_ID                         Action_Name permit Driver_Code_Primary Proposed_permit_other
08WW102206 Abbotsbury WRC - Phosphorus Removal 040001               HD_ND              0.20kg/d

in the Poole Harbour catchment? False


In [14]:
# Seasonality: the header advertises a convention the cells never use -- while the register DOES.
seasonal_headers = [c for c in PROPOSED if "Summer" in c or "summer" in c]
print("Columns whose header declares a seasonal convention:")
for c in seasonal_headers:
    print("   ", c)
marks = cells[cells.cell.str.contains(r"\bS\s*=|\bW\s*=|summer|winter", case=False, regex=True, na=False)]
print(f"\nCells actually using it, across all of Wessex Water: {len(marks)}")
print("\nMeanwhile the register carries genuinely seasonal limits -- permit 040067:")
print(sw[(sw.PERMIT_REF == "040067") & (sw.VERSION == "5")][
    ["PERMIT_REF","VERSION","DETE_CODE","DETE","MONTH_FROM","MONTH_TO","CODE_1","VAL_1","UNITS"]
].sort_values(["DETE_CODE","MONTH_FROM"]).to_string(index=False))


Columns whose header declares a seasonal convention:
    Proposed_BOD_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable
    Proposed_NH3_permit_95%ile(mg/l)(S=Summer;W=Winter)_plus_Upper_Tiers_where_applicable
    Proposed_P_annual_average_permit_mg/l_(S=summer)/backstop_limit(mg/l)

Cells actually using it, across all of Wessex Water: 0

Meanwhile the register carries genuinely seasonal limits -- permit 040067:
PERMIT_REF VERSION DETE_CODE                                                              DETE MONTH_FROM MONTH_TO        CODE_1 VAL_1               UNITS
    040067       5      0085                                                   BOD : 5 Day ATU         05       10 95 PERCENTILE    15 MILLIGRAM PER LITRE
    040067       5      0085                                                   BOD : 5 Day ATU         11       04 95 PERCENTILE    20 MILLIGRAM PER LITRE
    040067       5      0111                                          Ammoniacal Nitrogen as N  

A current limit here is seasonal — ammonia 5 mg/l May–October, 10 mg/l November–April. The proposed
limit has a header convention for expressing that and no value that uses it, so a proposed limit cannot
say what a current limit routinely says.


In [15]:
# The chemical family columns: a parameter family, not a determinand.
chem = [c for c in PROPOSED if "Chemical" in c]
print("Columns naming a parameter FAMILY rather than a determinand:")
def filled(frame, col):
    v = frame[col]
    return v.notna() & v.astype(str).str.strip().ne("")

for c in chem:
    print(f"   {int(filled(wq, c).sum()):>4} Wessex values   {c}")
print("\nSample values -- a number and a unit, with no substance anywhere:")
any_chem = wq[pd.concat([filled(wq, c) for c in chem], axis=1).any(axis=1)]
print(any_chem[["Action_ID","permit"] + chem].head(5).to_string(index=False))
print("\nin the Poole Harbour catchment:",
      int(sum(filled(has_limit, c).sum() for c in chem)), "values")
print("\nThe column fixes a unit and a statistic but names a parameter FAMILY, so a value parsed")
print("from it has a number, a unit and no analyte. Nothing in the row says which chemical.")


Columns naming a parameter FAMILY rather than a determinand:
     12 Wessex values   Proposed_Chemical_permit_99%ile_LUT_(ug/l)
      1 Wessex values   Proposed_Chemical_permit_95%ile_LUT_(ug/l)
      4 Wessex values   Proposed_Chemical_annual_average_permit_conditions_(ug/l)

Sample values -- a number and a unit, with no substance anywhere:
 Action_ID permit Proposed_Chemical_permit_99%ile_LUT_(ug/l) Proposed_Chemical_permit_95%ile_LUT_(ug/l) Proposed_Chemical_annual_average_permit_conditions_(ug/l)
08WW100220 071220                                                                                                                                            13.5
08WW100221 101456                     0.00044230000000000002                                                                                                     
08WW100222 102172                     0.00021626999999999999                                                                                                     
08WW1002

---
## C.3.3 Competing proposed limits for one permit and substance, with no stated precedence

> *"Permit 401050 is targeted by a Habitats Directive action proposing 0.25 mg/l phosphorus and by a
> UWWTR action proposing 2 mg/l."*


In [16]:
p_col = [c for c in PROPOSED if c.startswith("Proposed_P_annual")][0]
dorchester = has_limit[has_limit.permit == "401050"]
dorchester[["Action_ID","Action_Name","Driver_Code_Primary","Completion_Date", p_col, other]]


,Action_ID,Action_Name,Driver_Code_Primary,Completion_Date,Proposed_P_annual_average_permit_mg/l_(S=summer)/backstop_limit(mg/l),Proposed_permit_other
16484,08WW102104,Dorchester WRC - Phosphorus & Nitrogen Removal,HD_IMP_NN,2030-03-31,0.25,N 10mg/l
16491,08WW102200,Dorchester WRC - Nitrogen Permit (UWWTR),U_IMP1,2030-03-31,,N 15mg/l
16492,08WW102201,Dorchester WRC - Phosphorus Permit (UWWTR),U_IMP2,2030-05-13,2,


In [17]:
drivers = pd.read_excel(RAW / "PR24 WINEP National Dataset.xlsx",
                       sheet_name="Driver Codes Description", header=None)
codes = ["HD_IMP_NN", "U_IMP1", "U_IMP2"]
mask = drivers.apply(lambda r: r.astype(str).str.strip().isin(codes).any(), axis=1)
found = drivers[mask].dropna(axis=1, how="all")
print("What the workbook's own driver-code sheet says about the three drivers in play:\n")
print(found.to_string(index=False, header=False) if len(found) else
      "-- none of the three codes appears in the Driver Codes Description sheet --")


What the workbook's own driver-code sheet says about the three drivers in play:

Habitat Regulations \n(Previously known as Habitats Directive) HD_IMP_NN                                                                                                                                                                                                                                                                                                               Actions to reduce total phosphorus and/or total nitrogen levels to the Technically Achievable Limit (TAL) from discharges which drain to catchments where Nutrient Neutrality is advised.   *NEW* Statutory 2030-03-31 00:00:00
                       Urban Waste Water treatment Regulations    U_IMP1 Actions to improve discharges from agglomerations that, through population growth, have crossed the population thresholds in the UWWTR and therefore must achieve more stringent UWWTR requirements. This includes newly qualifying discharges (from ag

In [18]:
# Every (permit, substance) pair in the catchment carrying more than one proposal.
pairs = []
for col in [p_col, other, iron_col]:
    sub = has_limit[has_limit[col].notna() & has_limit[col].astype(str).str.strip().ne("")]
    for permit, grp in sub.groupby("permit"):
        if len(grp) > 1:
            pairs.append({"permit": permit, "column": col.replace("Proposed_", "")[:28],
                          "proposals": len(grp),
                          "values": " | ".join(grp[col].astype(str)),
                          "drivers": " | ".join(grp.Driver_Code_Primary.astype(str))})
pd.DataFrame(pairs)


,permit,column,proposals,values,drivers
0,401050,P_annual_average_permit_mg/l,2,0.25 | 2,HD_IMP_NN | U_IMP2
1,401747,P_annual_average_permit_mg/l,2,0.25 | 2,HD_IMP_NN | U_IMP1
2,401050,permit_other,2,N 10mg/l | N 15mg/l,HD_IMP_NN | U_IMP1
3,401050,Iron_/_Aluminium_permit_limi,2,No change from current | No change from current,HD_IMP_NN | U_IMP2
4,401747,Iron_/_Aluminium_permit_limi,2,No change from current | No change from current,HD_IMP_NN | U_IMP1


Two permits, five (permit, substance) pairs, each carrying proposals from two different regulatory
drivers. The Habitats Directive values are the tighter ones throughout. On phosphorus they also complete
earlier (2030-03-31 against the UWWTR action's 2030-05-13); on nitrogen both actions complete on
2030-03-31, so the dates separate nothing and only the values do. Either way the reading — alternatives,
not a phased sequence — has to be *inferred* from driver codes and completion dates. The dataset states no precedence, and `Driver_Code_Primary` is the only thing
distinguishing the two rows.

Loaded faithfully, this reads as a permit carrying two simultaneous phosphorus limits. Deriving the
effective one needs a most-stringent rule, and that rule needs a way to compare across statistics — an
annual average against a 95th percentile — which the source does not supply.
